In [1]:
# %pip install matplotlib numpy pandas seaborn scikit-learn

# Tải lên dữ liệu sau khi crawl

Thực hiện group_by theo tên đập và sort theo thời điểm

In [5]:
# Làm lại theo yêu cầu: KHÔNG dùng display_dataframe_to_user.
# Tạo df_ThuyDien đã sort và lưu ra CSV để tải về.
import pandas as pd

csv_in = "./data_thuydien/data_thuydien.csv"
csv_out = "./data_thuydien/df_ThuyDien_sorted.csv"

# --- Load CSV (thử nhiều encoding phổ biến ở VN)
encodings = ["utf-8", "utf-8-sig", "cp1258", "latin1"]
last_err = None
for enc in encodings:
    try:
        df_raw = pd.read_csv(csv_in, encoding=enc)
        break
    except Exception as e:
        last_err = e
else:
    raise last_err

# --- Chuẩn hoá tên cột (strip) và ánh xạ không phân biệt hoa thường
df_raw.columns = [c.strip() for c in df_raw.columns]
norm_map = {c.casefold().strip(): c for c in df_raw.columns}

tenho_col = norm_map.get("tên hồ") or norm_map.get("ten ho")
thoidiem_col = norm_map.get("thời điểm") or norm_map.get("thoi diem")

if not tenho_col or not thoidiem_col:
    raise ValueError(f"Không tìm thấy cột 'Tên hồ' hoặc 'Thời điểm'. Columns actual: {list(df_raw.columns)}")

# --- Ép kiểu datetime
df_work = df_raw.copy()
df_work[thoidiem_col] = pd.to_datetime(df_work[thoidiem_col], dayfirst=True, errors="coerce")
df_work = df_work.dropna(subset=[thoidiem_col])

# --- Sort theo "Tên hồ" rồi "Thời điểm"
df_ThuyDien = df_work.sort_values([tenho_col, thoidiem_col]).reset_index(drop=True)

# --- Lưu ra CSV
df_ThuyDien.to_csv(csv_out, index=False, encoding="utf-8-sig")

# --- In tóm tắt và preview (text)
print("rows_total_after_clean:", len(df_ThuyDien))
print("unique_reservoirs:", df_ThuyDien[tenho_col].nunique())
print("first_timestamp:", df_ThuyDien[thoidiem_col].min())
print("last_timestamp:", df_ThuyDien[thoidiem_col].max())
print("\nPreview (10 dòng đầu):")
print(df_ThuyDien.head(10).to_string(index=False))


rows_total_after_clean: 55440
unique_reservoirs: 5
first_timestamp: 2022-01-01 01:00:00
last_timestamp: 2025-10-17 23:00:00

Preview (10 dòng đầu):
  Tên hồ           Thời điểm  Mực nước  thượng lưu (m)  Mực nước  dâng bình thường (m)  Mực nước  chết (m)  Lưu lượng  đến hồ  (m3/s)  Tổng lượng xả  (m3/s)  Tổng lượng xả  qua đập  tràn (m3/s)  Tổng lượng xả  qua nhà  máy (m3/s)  Số cửa  xả sâu  Số cửa  xả mặt
Bản Chát 2022-01-01 01:00:00                    474.18                             475                 431                       44.5                    0.0                                  0.0                                 0.0               0               0
Bản Chát 2022-01-01 05:00:00                    474.19                             475                 431                       44.5                    0.0                                  0.0                                 0.0               0               0
Bản Chát 2022-01-01 07:00:00                    474.19            

# Tách thành 5 file - 5 đập riêng biệt

In [5]:
# Tách df_ThuyDien thành 5 file CSV theo từng "Tên hồ"
import os
import re
import pandas as pd
import unicodedata

base_dir = "./data_thuydien/"
out_files = []

def slugify(text: str) -> str:
    # Loại dấu tiếng Việt + ký tự đặc biệt -> tên file an toàn
    text = unicodedata.normalize("NFKD", text)
    text = "".join([c for c in text if not unicodedata.combining(c)])
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "unknown"

# 1) Lấy df_ThuyDien đã có; fallback đọc file đã lưu
try:
    df_ThuyDien  # check existence
    df = df_ThuyDien.copy()
except NameError:
    csv_sorted = os.path.join(base_dir, "df_ThuyDien_sorted.csv")
    df = pd.read_csv(csv_sorted, encoding="utf-8-sig")
    # Đảm bảo cột thời gian đúng kiểu
    cols = [c.strip() for c in df.columns]
    df.columns = cols
    norm_map = {c.casefold().strip(): c for c in df.columns}
    thoidiem_col = norm_map.get("thời điểm") or norm_map.get("thoi diem")
    if thoidiem_col:
        df[thoidiem_col] = pd.to_datetime(df[thoidiem_col], dayfirst=True, errors="coerce")
        df = df.dropna(subset=[thoidiem_col])

# 2) Xác định tên cột "Tên hồ"
df.columns = [c.strip() for c in df.columns]
norm_map = {c.casefold().strip(): c for c in df.columns}
tenho_col = norm_map.get("tên hồ") or norm_map.get("ten ho")
if not tenho_col:
    raise ValueError(f"Không tìm thấy cột 'Tên hồ'. Columns: {list(df.columns)}")

# 3) Tạo 1 file cho mỗi hồ
unique_reservoirs = list(pd.Series(df[tenho_col].unique()).dropna().astype(str).sort_values())
for res in unique_reservoirs:
    sub = df[df[tenho_col] == res].copy()
    fname = f"ThuyDien_{slugify(res)}.csv"
    fpath = os.path.join(base_dir, fname)
    sub.to_csv(fpath, index=False, encoding="utf-8-sig")
    out_files.append((res, len(sub), fpath))

# 4) In tóm tắt kết quả
print("Đã tạo các file sau:")
for res, nrows, fpath in out_files:
    print(f"- {res}: {nrows} dòng -> {fpath}")


Đã tạo các file sau:
- Bản Chát: 11088 dòng -> ./data_thuydien/ThuyDien_ban_chat.csv
- Huội Quảng: 11088 dòng -> ./data_thuydien/ThuyDien_huoi_quang.csv
- Hòa Bình: 11088 dòng -> ./data_thuydien/ThuyDien_hoa_binh.csv
- Sơn La: 11088 dòng -> ./data_thuydien/ThuyDien_son_la.csv
- Thác Bà: 11088 dòng -> ./data_thuydien/ThuyDien_thac_ba.csv


In [16]:
# Tách df_ThuyDien thành 5 file CSV theo từng "Tên hồ"
import os
import re
import pandas as pd
import unicodedata

base_dir = "./data_thuydien/"
out_files = []

df_ThuyDien = pd.read_csv(os.path.join(base_dir, "df_ThuyDien_reslotted.csv"), encoding="utf-8-sig")

def slugify(text: str) -> str:
    # Loại dấu tiếng Việt + ký tự đặc biệt -> tên file an toàn
    text = unicodedata.normalize("NFKD", text)
    text = "".join([c for c in text if not unicodedata.combining(c)])
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text or "unknown"

# 1) Lấy df_ThuyDien đã có; fallback đọc file đã lưu
try:
    df_ThuyDien  # check existence
    df = df_ThuyDien.copy()
except NameError:
    csv_sorted = os.path.join(base_dir, "df_ThuyDien_reslotted.csv")
    df = pd.read_csv(csv_sorted, encoding="utf-8-sig")
    # Đảm bảo cột thời gian đúng kiểu
    cols = [c.strip() for c in df.columns]
    df.columns = cols
    norm_map = {c.casefold().strip(): c for c in df.columns}
    thoidiem_col = norm_map.get("thời điểm") or norm_map.get("thoi diem")
    if thoidiem_col:
        df[thoidiem_col] = pd.to_datetime(df[thoidiem_col], dayfirst=True, errors="coerce")
        df = df.dropna(subset=[thoidiem_col])

# 2) Xác định tên cột "Tên hồ"
df.columns = [c.strip() for c in df.columns]
norm_map = {c.casefold().strip(): c for c in df.columns}
tenho_col = norm_map.get("tên hồ") or norm_map.get("ten ho")
if not tenho_col:
    raise ValueError(f"Không tìm thấy cột 'Tên hồ'. Columns: {list(df.columns)}")

# 3) Tạo 1 file cho mỗi hồ
unique_reservoirs = list(pd.Series(df[tenho_col].unique()).dropna().astype(str).sort_values())
for res in unique_reservoirs:
    sub = df[df[tenho_col] == res].copy()
    fname = f"ThuyDien_{slugify(res)}.csv"
    fpath = os.path.join(base_dir, fname)
    sub.to_csv(fpath, index=False, encoding="utf-8-sig")
    out_files.append((res, len(sub), fpath))

# 4) In tóm tắt kết quả
print("Đã tạo các file sau:")
for res, nrows, fpath in out_files:
    print(f"- {res}: {nrows} dòng -> {fpath}")


Đã tạo các file sau:
- Bản Chát: 11088 dòng -> ./data_thuydien/ThuyDien_ban_chat.csv
- Huội Quảng: 11088 dòng -> ./data_thuydien/ThuyDien_huoi_quang.csv
- Hòa Bình: 11088 dòng -> ./data_thuydien/ThuyDien_hoa_binh.csv
- Sơn La: 11088 dòng -> ./data_thuydien/ThuyDien_son_la.csv
- Thác Bà: 11088 dòng -> ./data_thuydien/ThuyDien_thac_ba.csv


# Xử lý lại dữ liệu bị trùng lặp giờ

In [7]:
import pandas as pd

# ========= CẤU HÌNH =========
PATH_IN  = "./data_thuydien/df_ThuyDien_sorted.csv"
PATH_OUT = "./data_thuydien/df_ThuyDien_reslotted.csv"
SLOTS = [2,5,8,11,14,17,20,23]  # giờ mục tiêu trong ngày

# ========= 1) ĐỌC & CHUẨN HÓA =========
df = pd.read_csv(PATH_IN, encoding="utf-8-sig")
df.columns = [c.replace("\ufeff", "").strip() for c in df.columns]

# Xác định tên cột (không phân biệt hoa/thường, có/không dấu)
def pick(colnames, *cands):
    cand_norm = {c.casefold().strip(): c for c in colnames}
    for key in cands:
        if key in cand_norm: 
            return cand_norm[key]
    raise KeyError(f"Không tìm thấy cột trong {cands}")

tenho_col    = pick(df.columns, "tên hồ", "ten ho")
thoidiem_col = pick(df.columns, "thời điểm", "thoi diem")

n0 = len(df)

# ========= 2) ÉP DATETIME THEO FORMAT CỐ ĐỊNH =========
# Dữ liệu bạn đã nói ở dạng 'YYYY-MM-DD HH:MM:SS'
dt = pd.to_datetime(df[thoidiem_col], format="%Y-%m-%d %H:%M:%S", errors="coerce")
nat_cnt = dt.isna().sum()
if nat_cnt > 0:
    # Nếu vẫn có NaT -> có ký tự lạ/chuỗi lệch; fallback linh hoạt một lần nữa:
    dt_fallback = pd.to_datetime(df[thoidiem_col], errors="coerce")  # auto-parse ISO/biến thể
    dt = dt.fillna(dt_fallback)
    nat_cnt = dt.isna().sum()
    if nat_cnt > 0:
        # Gợi ý: in vài giá trị lỗi để bạn xử lý bằng mắt nếu cần
        print("[CẢNH BÁO] Vẫn còn NaT sau parse:", nat_cnt)
        print("Mẫu lỗi:", df.loc[dt.isna(), thoidiem_col].drop_duplicates().head(10).tolist())

df[thoidiem_col] = dt
# KHÔNG dropna ở đây, để không mất dòng; reslot vẫn thực hiện cho phần parse được.

# ========= 3) TẠO CỘT NGÀY VÀ RESLOT THEO (TÊN HỒ, NGÀY) =========
df["_date"] = df[thoidiem_col].dt.floor("D")  # NaT -> NaT, nhóm riêng sẽ giữ nguyên

def assign_slots_for_group(g: pd.DataFrame) -> pd.DataFrame:
    out = g.copy()
    # Nếu cả nhóm không có ngày hợp lệ (toàn NaT), trả nguyên nhóm
    if out["_date"].notna().sum() == 0:
        out["Thời điểm_gốc"] = out[thoidiem_col]
        return out
    # Lấy ngày chuẩn đầu tiên (không NaT)
    day0 = out.loc[out["_date"].notna(), "_date"].iloc[0]
    n = len(out)
    schedule = []
    for i in range(n):
        day_offset = i // len(SLOTS)
        hour = SLOTS[i % len(SLOTS)]
        schedule.append(day0 + pd.Timedelta(days=day_offset, hours=hour))
    out["Thời điểm_gốc"] = out[thoidiem_col]
    out[thoidiem_col] = schedule
    return out

# Không dùng include_groups để tránh phụ thuộc phiên bản pandas; cũng không sort lại.
parts = []
for (res_name, day), g in df.groupby([tenho_col, "_date"], sort=False):
    parts.append(assign_slots_for_group(g))

df_out = pd.concat(parts, axis=0, ignore_index=True)

# ========= 4) KIỂM TRA & LƯU =========
assert len(df_out) == n0, f"Số dòng thay đổi (không hợp lệ): trước {n0}, sau {len(df_out)}"

# Kiểm tra hợp lệ slot (chỉ với các dòng parse được)
ok_mask = df_out[thoidiem_col].notna()
hours_ok   = df_out.loc[ok_mask, thoidiem_col].dt.hour.isin(SLOTS).all()
minutes_ok = (df_out.loc[ok_mask, thoidiem_col].dt.minute == 0).all()
seconds_ok = (df_out.loc[ok_mask, thoidiem_col].dt.second == 0).all()

changed = int((df_out["Thời điểm_gốc"] != df_out[thoidiem_col]).sum())
df_out = df_out.drop(columns=["_date"])
df_out.to_csv(PATH_OUT, index=False, encoding="utf-8-sig")

print("TÓM TẮT RESLOT:")
print(f"- Số dòng đầu vào:  {n0}")
print(f"- Dòng đổi giờ:      {changed}")
print(f"- Hợp lệ slot (giờ): {bool(hours_ok and minutes_ok and seconds_ok)}")
print(f"- File đã lưu:       {PATH_OUT}")

TÓM TẮT RESLOT:
- Số dòng đầu vào:  55440
- Dòng đổi giờ:      14195
- Hợp lệ slot (giờ): True
- File đã lưu:       ./data_thuydien/df_ThuyDien_reslotted.csv
